# RQ1 — Ministry × Topic × Party × Time

**Algorithms (syllabus mapping)**
- W3 MinHash/LSH — duplicate önerge detection
- W6 FP-Growth — frequent (party, ministry, topic) co-occurrence
- W10 ALS/SVD — latent ministry-party factorisation
- LDA — topic modelling on full text (Spark MLlib)

**Inputs**: `silver_yazili_soru_clean` Delta table.

**Outputs (Gold)**:
- `ministry_topic_party_year` — heatmap source
- `duplicate_clusters` — MinHash near-duplicate önerge groups

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from spark_utils import get_spark, read_delta, write_delta, TABLES
from pyspark.sql import functions as F
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, MinHashLSH, HashingTF
from pyspark.ml.clustering import LDA
from pyspark.ml.fpm import FPGrowth

spark = get_spark('rq1', memory='6g')
spark.sparkContext.setLogLevel('WARN')
silver = read_delta(spark, TABLES['silver_yazili_soru_clean']).cache()
mp = read_delta(spark, TABLES['silver_mp_party']).select('mp_guid', F.col('party').alias('mp_party'), F.col('province').alias('mp_province'))
edges = read_delta(spark, TABLES['silver_cosign_edges'])
print('Silver rows:', silver.count())

## 1. Join party onto önerge via primary signer (sahip_isim+il match)

In [ ]:
# Join through mp_to_guids edge table to get authoritative party + province
joined = (
    silver.join(edges, edges.onerge_guid == silver.guid, 'left')
          .join(mp, edges.mp_guid == mp.mp_guid, 'left')
)
joined.select('guid', 'mp_party', 'muhatap_bakanlık', 'year', 'ozet_norm').show(5, truncate=50)

## 2. LDA topic modelling on full text

In [ ]:
TURKISH_STOPWORDS = [
    've', 'ile', 'için', 'bir', 'bu', 'da', 'de', 'ki', 'mi', 'çok',
    'gibi', 'kadar', 'her', 'olarak', 'üzere', 'olan', 'olarak', 'sayın',
    'soru', 'önerge', 'milletvekili', 'bakanlık', 'bakanı', 'tbmm',
    'türkiye', 'cumhuriyeti', 'genel', 'müdürlüğü', 'kurumu',
    'ne', 'ne kadar', 'nasıl', 'kim', 'nedir', 'mıdır',
]
tokenizer = Tokenizer(inputCol='full_text', outputCol='tokens')
stops = StopWordsRemover(inputCol='tokens', outputCol='clean_tokens', stopWords=TURKISH_STOPWORDS)
cv = CountVectorizer(inputCol='clean_tokens', outputCol='tf', vocabSize=5000, minDF=5)
idf = IDF(inputCol='tf', outputCol='features')
lda = LDA(k=20, maxIter=15, featuresCol='features', seed=42)

from pyspark.ml import Pipeline
pipe = Pipeline(stages=[tokenizer, stops, cv, idf, lda])
model = pipe.fit(joined.filter(F.length('full_text') > 50))
vocab = model.stages[2].vocabulary
topics = model.stages[-1].describeTopics(maxTermsPerTopic=10).collect()
for row in topics:
    words = [vocab[i] for i in row.termIndices]
    print(f'Topic {row.topic}: {" ".join(words)}')

In [ ]:
scored = model.transform(joined)
topic_assigned = scored.withColumn('topic', F.expr('CAST(array_position(topicDistribution.values, array_max(topicDistribution.values)) - 1 AS INT)'))
# Fallback if topicDistribution dense vector
from pyspark.ml.linalg import DenseVector, VectorUDT
from pyspark.sql.types import IntegerType
@F.udf(IntegerType())
def argmax(v):
    if v is None: return -1
    arr = v.toArray() if hasattr(v, 'toArray') else v
    return int(arr.argmax())
topic_assigned = scored.withColumn('topic', argmax('topicDistribution'))
topic_assigned.groupBy('topic').count().orderBy(F.desc('count')).show()

## 3. FP-Growth — (party, ministry, topic) co-occurrence

In [ ]:
baskets = (
    topic_assigned
    .filter(F.col('mp_party').isNotNull() & F.col('muhatap_bakanlık').isNotNull())
    .withColumn('item_party', F.concat(F.lit('PARTY:'), F.col('mp_party')))
    .withColumn('item_min', F.concat(F.lit('MIN:'), F.col('muhatap_bakanlık')))
    .withColumn('item_topic', F.concat(F.lit('TOPIC:'), F.col('topic')))
    .select(F.array('item_party', 'item_min', 'item_topic').alias('items'), 'guid')
)
fp = FPGrowth(itemsCol='items', minSupport=0.005, minConfidence=0.3)
fp_model = fp.fit(baskets)
print('Frequent itemsets:')
fp_model.freqItemsets.orderBy(F.desc('freq')).show(20, truncate=80)
print('Top association rules:')
fp_model.associationRules.orderBy(F.desc('lift')).show(20, truncate=80)

## 4. MinHash/LSH — duplicate önerge detection

In [ ]:
from pyspark.ml.feature import HashingTF
feat_tokens = StopWordsRemover(inputCol='tokens', outputCol='clean_tokens2', stopWords=TURKISH_STOPWORDS).transform(
    Tokenizer(inputCol='ozet_norm', outputCol='tokens').transform(
        joined.filter(F.length('ozet_norm') > 20).select('guid', 'ozet_norm')
    )
)
htf = HashingTF(inputCol='clean_tokens2', outputCol='vec', numFeatures=4096, binary=True)
vecs = htf.transform(feat_tokens).filter(F.size('clean_tokens2') > 3)
lsh = MinHashLSH(inputCol='vec', outputCol='hashes', numHashTables=8)
lsh_model = lsh.fit(vecs)
# self-join with jaccard distance <= 0.2 ↔ similarity >= 0.8
dups = lsh_model.approxSimilarityJoin(vecs, vecs, 0.2, distCol='dist') \
    .filter('datasetA.guid < datasetB.guid') \
    .select(F.col('datasetA.guid').alias('g1'), F.col('datasetB.guid').alias('g2'), 'dist')
print('Near-duplicate pairs:', dups.count())
dups.orderBy('dist').show(10, truncate=False)

## 5. Gold aggregates

In [ ]:
gold = (
    topic_assigned
    .filter('mp_party IS NOT NULL AND muhatap_bakanlık IS NOT NULL')
    .groupBy('muhatap_bakanlık', 'mp_party', 'topic', 'year')
    .agg(F.count('*').alias('n'))
)
write_delta(gold, TABLES['gold_ministry_topic_party_year'], partition_by=['year'])
write_delta(dups, TABLES['gold_ministry_topic_party_year'].parent / 'duplicate_clusters')
print('Gold written.')